In [1]:
# Build the mesh scaling file that the clutter generator uses to normalize object sizes.
from mesh_scale_bounds import scale_dataset_and_save_json

# `root_dir`: mesh dataset folder scanned recursively for supported mesh files.
# `output_json`: destination JSON; each record stores mesh path, AABB stats, and scale factor.
# `target_min_axis_max`: after scaling, the smallest AABB axis must be <= this value.
# `target_max_axis_strict`: after scaling, the largest AABB axis must stay strictly below this value.
results = scale_dataset_and_save_json(
    root_dir="mesh_dataset",
    output_json="mesh_scaling_results.json",
    target_min_axis_max=0.07,
    target_max_axis_strict=0.3,
)

# `results` is the in-memory copy of what was written to disk; preview a couple of entries.
print(f"Saved {len(results)} mesh records to mesh_scaling_results.json")
results[:2]


-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
Mesh: mesh_dataset/vase/vase_0001.off
AABB size xyz: [ 9.16513  9.16513 17.8735 ]
Diagonal: 22.078523910891324
Scale factor: 0.007637643983227735
Scaled AABB xyz: [0.07       0.07       0.13651143]
Scaled min axis: 0.07
Scaled max axis: 0.13651142973422092
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
Mesh: mesh_dataset/guitar/guitar_0008.off
AABB size xyz: [13.10452 44.50925  2.33   ]
Diagonal: 46.45676140232872
Scale factor: 0.00674017196425462
Scaled AABB xyz: [0.08832672 0.3        0.0157046 ]
Scaled min axis: 0.015704600676713264
Scaled max axis: 0.29999999899999996
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
Mesh: mesh_dataset/whippedCream/WhippedCream_25k_tex.obj
AABB size xyz: [ 59.04379845 218.01979446  59.98559952]
Diagonal: 233.70295903709726
Scale factor: 

[{'mesh_path': 'mesh_dataset/vase/vase_0001.off',
  'aabb_size_xyz': [9.16513, 9.16513, 17.8735],
  'aabb_diagonal': 22.078523910891324,
  'min_axis_before': 9.16513,
  'max_axis_before': 17.8735,
  'scale_factor': 0.007637643983227735,
  'aabb_size_xyz_after_scaling': [0.07, 0.07, 0.13651142973422092],
  'min_axis_after': 0.07,
  'max_axis_after': 0.13651142973422092,
  'constraints': {'min_axis_after_lte': 0.07, 'max_axis_after_lt': 0.3},
  'constraints_satisfied': True},
 {'mesh_path': 'mesh_dataset/guitar/guitar_0008.off',
  'aabb_size_xyz': [13.10452, 44.50925, 2.33],
  'aabb_diagonal': 46.45676140232872,
  'min_axis_before': 2.33,
  'max_axis_before': 44.50925,
  'scale_factor': 0.00674017196425462,
  'aabb_size_xyz_after_scaling': [0.08832671830901397,
   0.29999999899999996,
   0.015704600676713264],
  'min_axis_after': 0.015704600676713264,
  'max_axis_after': 0.29999999899999996,
  'constraints': {'min_axis_after_lte': 0.07, 'max_axis_after_lt': 0.3},
  'constraints_satisfied

In [2]:
# Load `robotic` and the mesh-clutter generator built on top of the Panda table scene.
import robotic as ry

from PandaTableMeshClutterGenerator import PandaTableMeshClutterGenerator

# Configure one generator instance.
# `base_scene_file`: base Panda + table scene used as the starting world.
# `dataset_dir`: root folder containing one subfolder per object category / mesh asset.
# `scaling_json_path`: precomputed mesh scaling file from the previous cell.
# `output_dir`: where generated `.g` scenes can be saved later.
# `table_frame_name`: name of the support-surface frame used for placement logic.
# `gap`: spacing margin used when sampling placements on the table.
# `spawn_height`: initial drop height used before simulation settles the objects.
# `seed`: fixes the RNG so the same setup can be reproduced.
# `object_mass`: mass assigned to spawned clutter objects.
# `table_shape_size`: table dimensions used for region and quarter calculations.
# `panda_base_relative_pos`: Panda base offset relative to the table/world setup.
# `target_alpha`: transparency of the final green `goal_pose_...` marker object.
# `target_center_jitter_ratio`: small random offset around the chosen target-quarter center.
# `clutter_mode`: controls how clutter is distributed across the table.
# `placement_candidate_count`: how many XY placement samples are evaluated per spawn.
# `hardnessOfTargetObject`: controls how early the hidden `goal_...` copy appears in clutter.
generator = PandaTableMeshClutterGenerator(
    base_scene_file=ry.raiPath("../rai-robotModels/scenarios/pandaSingle.g"),
    dataset_dir="mesh_dataset",
    scaling_json_path="mesh_scaling_results.json",
    output_dir="./generated_envs",
    table_frame_name="table",
    gap=0.04,
    spawn_height=0.42,
    seed=3,
    object_mass=0.1,
    table_shape_size=(1.6, 1.6, 0.08, 0.02),
    panda_base_relative_pos=(0.0, 0.0, 0.05),
    target_alpha=0.35,
    target_center_jitter_ratio=0.05,
    clutter_mode="high_clutter",   # "random", "low_clutter", or "high_clutter"
    placement_candidate_count=128,
    hardnessOfTargetObject=0.0,
)

# Generate one environment and keep refilling clutter until enough objects remain on the table.
# `num_objects`: target count of regular clutter objects (the goal copies are handled separately).
# `sim_seconds`: physics duration per settle/refill round.
# `sim_dt`: simulation time step.
# `max_refill_rounds`: stop after this many refill attempts even if some objects fell off.
# `xy_margin`: table-boundary margin used when deciding whether an object is still on the table.
# `z_tolerance`: acceptable height slack when classifying objects as on-table.
# `batch_spawn_count`: maximum number of clutter objects respawned in one refill round.
# `max_target_spawn_attempts`: retries for placing the target-related objects robustly.
# Returns: `C` (the final `ry.Config`) and `summary` (metadata about target choice and spawned objects).
C, summary = generator.create_environment_with_refill(
    num_objects=30,
    sim_seconds=10.0,
    sim_dt=0.01,
    max_refill_rounds=20,
    xy_margin=0.02,
    z_tolerance=0.15,
    batch_spawn_count=4,
    max_target_spawn_attempts=40,
)

# `summary` contains the selected target identity, clutter stats, frame prefixes, and per-object records.
print("\n--- Scene summary ---")
print("target_object_name       :", summary["target_object_name"])
print("target_mesh_basename     :", summary["target_mesh_basename"])
print("target_quarter_mode      :", summary["target_quarter_mode"])
print("clutter_mode             :", summary["clutter_mode"])
print("final_on_table           :", summary["final_on_table"])
print("final_off_table          :", summary["final_off_table"])
print("goal_clutter_prefix      :", summary["goal_clutter_object_prefix"])
print("goal_pose_prefix         :", summary["goal_pose_object_prefix"])

# Keep only surviving objects; `summary["objects"]` also tracks entries that were removed or fell off.
alive_objects = [obj for obj in summary["objects"] if obj["alive"]]
print("\nAlive objects:", len(alive_objects))
for obj in alive_objects:
    print(
        obj["role"],
        obj["object_name"],
        obj["mesh_basename"],
        obj["spawn_xy"],
        round(obj["theta_deg"], 2),
    )


# Open the interactive viewer for the generated scene.
C.view()




-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_core is trivial -> removing contact flag
-- WARNING:frame.cpp:createMeshes:2215(-1) shape tmp_mesh coll_c

0

In [3]:
# Save the current `ry.Config` to a `.g` scene file.
# The helper also rewrites mesh paths so the saved scene can be reloaded more reliably later.
from save_scene_to_g import save_config_as_g

saved_path = save_config_as_g(C, "generated_envs/mesh_clutter_high_30.g")
print(saved_path)


Saved: generated_envs/mesh_clutter_high_30.g
generated_envs/mesh_clutter_high_30.g


In [7]:
# Recreate a fresh config from the saved `.g` file.
# `cache_bust_mesh_paths=True` reloads meshes from a temp mirror to avoid stale texture/cache issues.
# `created` stores metadata for the mesh frames rebuilt during reload.
from rebuild_scene_from_g import recreate_scene_from_g

C_loaded, created = recreate_scene_from_g(
    scene_file="generated_envs/mesh_clutter_high_30.g",
    cache_bust_mesh_paths=True,
)

# View the reconstructed scene to verify that saving and reloading preserved the environment.
C_loaded.view()


0

In [5]:
C_loaded.getFrameNames()

['world',
 'table',
 'l_panda_base',
 'l_panda_joint1_origin',
 'l_panda_joint2_origin',
 'l_panda_joint3_origin',
 'l_panda_joint4_origin',
 'l_panda_joint5_origin',
 'l_panda_joint6_origin',
 'l_panda_joint7_origin',
 'l_panda_joint8_origin',
 'l_panda_joint8',
 'l_panda_hand_joint_origin',
 'l_panda_finger_joint1_origin',
 'l_panda_finger_joint2_origin',
 'l_panda_finger_joint2',
 'l_panda_coll0',
 'l_panda_coll1',
 'l_panda_coll3',
 'l_panda_coll5',
 'l_panda_coll2',
 'l_panda_coll4',
 'l_panda_coll6',
 'l_panda_coll7',
 'l_gripper',
 'l_palm',
 'l_finger1',
 'l_finger2',
 'cameraTop',
 'cameraWrist',
 'panda_collCameraWrist',
 'l_panda_link0',
 'l_panda_joint1',
 'l_panda_joint2',
 'l_panda_joint3',
 'l_panda_joint4',
 'l_panda_joint5',
 'l_panda_joint6',
 'l_panda_joint7',
 'l_panda_hand_joint',
 'l_panda_finger_joint1',
 'l_panda_rightfinger_0',
 'obj_0_mesh',
 'obj_1_mesh',
 'obj_2_mesh',
 'obj_3_mesh',
 'obj_4_mesh',
 'obj_5_mesh',
 'obj_7_mesh',
 'obj_8_mesh',
 'obj_10_mesh',